# Compress ResNet-56 (CIFAR-10) — complete Kaggle notebook

`ResNet-56 (FP32, 56 layers)  →  gradual prune + fine-tune  →  INT8 quantize  →  ONNX  →  verify against the original`

The ResNet-56 baseline (92%+ on CIFAR-10) ships inside the repo, so **no training step**.

**Before Run All** (right panel): *Accelerator → GPU T4*, *Internet → On*.
Optional but saves ~30 min: *Add Input → search "CIFAR-10"* and attach any CIFAR-10 dataset in the
original python-pickle format (it must contain a `cifar-10-batches-py` folder). Otherwise it downloads (170 MB).

Everything lands in `/kaggle/working/` — download `resnet56_compressed.zip` from the *Output* tab.

In [ ]:
# ======================= CONFIG =======================
PRUNE_RATIO = 0.5    # fraction of channels removed (0.5 channels ~ 75% fewer parameters)
STEPS       = 5      # gradual pruning steps
FT_EPOCHS   = 5      # fine-tune epochs after every step (25 total)
LOSS        = "dkd"  # "kd" | "dkd"
# ======================================================

In [ ]:
import os, subprocess, sys, json, shutil, pathlib
REPO = "https://github.com/Bhagyesh0603/Model-Compression-ToolKit..git"
if not os.path.exists("/kaggle/working/ck"):
    subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/ck"], check=True)
os.chdir("/kaggle/working/ck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> enable GPU in the right panel, then Restart & Run All")
from compresskit.zoo.data import resolve_root
print("CIFAR-10 location:", resolve_root(), "(attached dataset if under /kaggle/input, else it will download)")
assert os.path.exists("checkpoints/resnet56_fp32.pt"), "baseline checkpoint missing from the repo"

def run(*args):
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    return subprocess.run(args).returncode

## 1 · Compress: gradual prune + fine-tune → INT8 → evaluate the .onnx

In [ ]:
OUT = "/kaggle/working"
code_ = run(sys.executable, "-m", "compresskit.cli", "pipeline",
            "--model", "checkpoints/resnet56_fp32.pt",
            "--prune", PRUNE_RATIO, "--steps", STEPS, "--fine-tune-epochs", FT_EPOCHS, "--loss", LOSS,
            "--precision", "int8", "--allow-slow-int8",
            "--workdir", f"{OUT}/work", "--report", f"{OUT}/report.html", "--json", f"{OUT}/results.json")
assert code_ == 0, "pipeline failed - read the log above"

## 2 · Does the compressed model actually work? (original vs ONNX on 2000 real test images)

In [ ]:
r = json.load(open(f"{OUT}/results.json"))
final_onnx = r["stages"][-1]["artifact"]
shutil.copy(final_onnx, f"{OUT}/model.onnx")
shutil.copy("checkpoints/resnet56_fp32.pt", f"{OUT}/resnet56_fp32.pt")
run(sys.executable, "experiments/verify_onnx.py", "--model", "checkpoints/resnet56_fp32.pt",
    "--onnx", f"{OUT}/model.onnx", "--images", 2000)

## 3 · Summary and download bundle

In [ ]:
b, f, t = r["baseline"], r["stages"][-1], r["totals"]
print(f"accuracy : {b['accuracy']*100:.2f}% -> {f['accuracy']*100:.2f}%   (drop {t['accuracy_drop_pp']:.2f} pp)")
print(f"size     : {b['size_mb']:.2f} MB -> {f['size_mb']:.2f} MB   ({t['size_reduction']:.1f}x smaller)")
print(f"precision: {f['precision']}")
print("NOTE: latency in this run is Kaggle-CPU latency, not your device. Measure on the target: compresskit bench model.onnx")

import zipfile
with zipfile.ZipFile(f"{OUT}/resnet56_compressed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("model.onnx", "report.html", "results.json", "resnet56_fp32.pt"):
        z.write(f"{OUT}/{name}", name)
print("bundle:", f"{OUT}/resnet56_compressed.zip", f"({os.path.getsize(f'{OUT}/resnet56_compressed.zip')/1e6:.1f} MB)")
from IPython.display import IFrame, display
display(IFrame(f"{OUT}/report.html", width="100%", height=900))

## 4 · On your Mac (real speed check, once)

```bash
unzip resnet56_compressed.zip
python experiments/verify_onnx.py --model resnet56_fp32.pt --onnx model.onnx
compresskit bench model.onnx
```